<p>
<h1><b><center></center></b></h1>
<center><img src="https://drive.google.com/uc?id=1UJc1ci41G6ahJ7ProKvunUOIBcTXZ6ZG" align="center" width="550"></center>
</p>
<h1><b><center>Mecánica Celeste</center></b></h1>
<h2><b><center>Prof. Jorge I. Zuluaga</center></b></h1>
<h2><b><center>Proyecto del Curso</center></b><h2>
<h3><b><center>Disco de acreción alrededor de un agujero negro</center></b><h3>
<h7><center><i>Última actualización del profesor</b>: Viernes 20 de marzo de 2026, 2:00 pm</i></center><h7>
</p>

## Enunciado

<center>
<img src="https://cdn.sci.news/images/2021/02/image_9386-AT2019dsg.jpg" align="center" width="100%"></center>

Muchos objetos astrofísicos fascinantes (núcleos galácticos activos, hipernovas, binarias de rayos X) pueden explicarse con los fenómenos que ocurren en los denominados discos de acreción alrededor de agujeros negros. En este proyecto usaremos lo visto en el curso de Relatividad y Gravitación para estudiar esos fenómenos.

El objetivo de este trabajo es desarrollar simulaciones, cálculos útiles, predicciones sobre el comportamiento de las partículas en un disco de acreción. La idea es poner en práctica lo visto en el curso, tanto en las partes de relatividad especial como en las de relatividad general, para estudiar estos sistemas.

## Algunas ideas

Existen muchas maneras de aplicar la teoría que veremos en el curso en este problema y no queremos sesgar tu elección de los temas o cálculos que quieras escoger para aplicarla. Sin embargo aquí van algunas ideas de cálculos que se podrían hacer:
- Simulación del movimiento de muchas partículas alrededor del agujero negro usando, inicialmente, la dinámica newtoniana.
- Cálculo y representación gráfica de la luz emitida por el disco y observada desde distinto ángulos, incluyendo los efectos de beaming y desplazamiento espectral.
- Simulación del jet del agujero negro usando partículas cargadas moviéndose en el campo del jet.
- Cálculo de la trayectoria de fotones para visualizar el disco de acreción desde distintos ángulos.
- Cálculo de las trayectorias de partículas usando la métrica de Schwarzschild y de Kerr.

## Entregables

El entregable del proyecto es **un notebook final de Jupyter** con una descripción de la teoría básica que uses, los experimentos numéricos que hayas realizado y las conclusiones a las que llegues con esos experimentos. Por supuesto puedes desarrollar otros programas y notebooks paralelos, pero se revisará el notebook con el reporte final.

Adicionalmente se deberá entregar **un repositorio de GitHub** que tenga todos los archivos, datos, notebooks, programas usados para este propósito. El notebook debe estar alojado en el repositorio.

## Criterios de evaluación

Una vez entregues el proyecto el profesor realizará una revisión del mismo y te lo devolverá con observaciones. En la segunda revisión emitirá un concepto cuantitativo del proyecto. Los criterios a evaluar serán:

- Correcta descripción y aplicación de la teoría.
- Originalidad de los experimentos numéricos.
- Conclusiones derivadas de los experimentos.
- Organización y extensión del reporte final.
- Ritmo de actualizaciones del repositorio de GitHub.

## Para tener en cuenta

- La solución presentada debe ser estrictamente individual. Evite resolver la tarea en parejas o en grupos que puede conducir a códigos o soluciones idénticos o muy similares.
- Los métodos y herramientas para resolver el problema deben ser los vistos en clase. El uso de herramientas diferentes puede ser una buena práctica en el mundo académico o laboral, pero en un curso puede también ser un indicio de un mal uso de las *asesorías* externa o del uso inapropiado de herramientas de Inteligencia Artificial (IA).
- El notebook entregado debe tener todos los resultados y gráficos, calculados y a la vista.  También debe ejecutarse completamente con `Ejecutar Todo` sin producir ningún error (verifique antes de entregar).
- El notebook debe tener explicaciones detalladas para cada paso del procedimiento usando celdas de texto. No debe poner una celda de código sin explicarla. En caso de incluir ecuaciones debe usar $\LaTeX$.

In [1]:
!pip3 install -Uq pymcel rebound einsteinpy


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import rebound as rb
import einsteinpy.symbolic as es
from scipy.integrate import solve_ivp


Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


In [ ]:
G_SI = 6.67430e-11      # m³ kg⁻¹ s⁻²
c_SI = 299792458.0      # m/s
M_sun = 1.98847e30   # kg

In [ ]:
UM = M_sun
UL = G_SI * UM / c_SI**2   # 1476.6 m
UT = UL / c_SI                     # 4.925e-6 s

# En unidades relativistas
G = 1.0
C = 1.0
M = 1.0  # Masa del agujero negro en unidades solares

##### Ecuaciones de la geodésica en la métrica de Newton. Basado principalmente del notebook "3. Geodesicas de Galileo y Geodesicas de Newton"    
Ys = [T, x, y, z, Ut, Ux, Uy, Uz]

In [ ]:
# Ecuación de la geodésica (Del notebook 3.)
def edg_newton(tau, Ys, G, M, c):
    T, x, y, z, Ut, Ux, Uy, Uz = Ys

    r = np.sqrt(x**2 + y**2 + z**2)

    dT_dtau = Ut
    dx_dtau = Ux
    dy_dtau = Uy
    dz_dtau = Uz

    denom = (c**2 * r - 2 * G * M) * r**2

    dUt_dtau = -2 * G * M * Ut * (x * Ux + y * Uy + z * Uz) / denom
    dUx_dtau = -(G * M * x / (c**2 * r**3)) * Ut**2
    dUy_dtau = -(G * M * y / (c**2 * r**3)) * Ut**2
    dUz_dtau = -(G * M * z / (c**2 * r**3)) * Ut**2

    return [dT_dtau, dx_dtau, dy_dtau, dz_dtau,
            dUt_dtau, dUx_dtau, dUy_dtau, dUz_dtau]

In [ ]:
# ── Ecuación de la geodésica (idéntica al notebook) ───────────────────────────
def edg_newton(tau, Ys, G, M, c):
    T, x, y, z, Ut, Ux, Uy, Uz = Ys

    r = np.sqrt(x**2 + y**2 + z**2)

    dT_dtau = Ut
    dx_dtau = Ux
    dy_dtau = Uy
    dz_dtau = Uz

    denom = (c**2 * r - 2 * G * M) * r**2

    dUt_dtau = -2 * G * M * Ut * (x * Ux + y * Uy + z * Uz) / denom
    dUx_dtau = -(G * M * x / (c**2 * r**3)) * Ut**2
    dUy_dtau = -(G * M * y / (c**2 * r**3)) * Ut**2
    dUz_dtau = -(G * M * z / (c**2 * r**3)) * Ut**2

    return [dT_dtau, dx_dtau, dy_dtau, dz_dtau,
            dUt_dtau, dUx_dtau, dUy_dtau, dUz_dtau]

# ── Parámetros de la simulación ───────────────────────────────────────────────
N_particulas = 200                   # <-- cambia este número

rng = np.random.default_rng(42)

r_min = 5_000e3  / UL               # 5 000 km en unidades canónicas
r_max = 30_000e3 / UL               # 30 000 km en unidades canónicas

# Coordenadas esféricas iniciales
r0s    = rng.uniform(r_min, r_max, N_particulas)   # radio
thetas = rng.uniform(0, np.pi,     N_particulas)   # polar (0 = norte, π = sur)
phis   = rng.uniform(0, 2*np.pi,   N_particulas)   # azimutal

# Período kepleriano del radio medio → tiempo de integración
r_medio  = 0.5 * (r_min + r_max)
T_kepler = 2 * np.pi * np.sqrt(r_medio**3 / (G * M))
taus = np.linspace(0, 0.98 * T_kepler, 1000)

# ── Integración ───────────────────────────────────────────────────────────────
soluciones = []

for i in range(N_particulas):
    r0    = r0s[i]
    theta = thetas[i]
    phi   = phis[i]

    # Esféricas → cartesianas
    x0 = r0 * np.sin(theta) * np.cos(phi)
    y0 = r0 * np.sin(theta) * np.sin(phi)
    z0 = r0 * np.cos(theta)

    # Velocidad circular tangencial (dirección azimutal)
    vc  = np.sqrt(G * M / r0)
    vx0 = -vc * np.sin(phi)
    vy0 =  vc * np.cos(phi)
    vz0 =  0.0

    # 4-velocidad inicial (igual que en el notebook)
    g00 = 1 - 2*G*M / (C**2 * r0)
    g11 = g22 = g33 = -1

    Ut0 = 1 / np.sqrt(g00 + g11*vx0**2 + g22*vy0**2 + g33*vz0**2)
    Ux0 = vx0 * Ut0 / C
    Uy0 = vy0 * Ut0 / C
    Uz0 = vz0 * Ut0 / C

    Ys0 = [0.0, x0, y0, z0, Ut0, Ux0, Uy0, Uz0]

    sol = solve_ivp(
        edg_newton,
        (taus[0], taus[-1]),
        Ys0,
        t_eval=taus,
        args=(G, M, C),
        method='Radau'
    )

    soluciones.append(sol)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_particulas} partículas integradas")

print("Integración completada.")

# ── Extraer posiciones finales en km ─────────────────────────────────────────
xs_km = np.array([sol.y[1, -1] * UL / 1e3 for sol in soluciones])
ys_km = np.array([sol.y[2, -1] * UL / 1e3 for sol in soluciones])
zs_km = np.array([sol.y[3, -1] * UL / 1e3 for sol in soluciones])

radios_finales = np.sqrt(xs_km**2 + ys_km**2 + zs_km**2)
r_min_km = radios_finales.min()
r_max_km = radios_finales.max()

cmap    = plt.cm.plasma
norm    = plt.Normalize(vmin=r_min_km, vmax=r_max_km)
colores = cmap(norm(radios_finales))

# ── Animación 3D con rotación ─────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 8), facecolor='black')
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('black')
fig.patch.set_facecolor('black')

ax.scatter(xs_km, ys_km, zs_km, c=colores, s=4, depthshade=False)
ax.scatter([0], [0], [0], color='white', s=60, zorder=10)

sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.1)
cbar.set_label('Radio  [km]', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

lim = r_max_km * 1.1
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)
ax.set_xlabel('x [km]', color='white', labelpad=6)
ax.set_ylabel('y [km]', color='white', labelpad=6)
ax.set_zlabel('z [km]', color='white', labelpad=6)
ax.tick_params(colors='white')
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor('#222222')
ax.yaxis.pane.set_edgecolor('#222222')
ax.zaxis.pane.set_edgecolor('#222222')
ax.set_title(f'Geodésicas newtonianas — {N_particulas} partículas',
             color='white', pad=10)

def rotar(frame):
    ax.view_init(elev=20, azim=frame)
    return []

ani = animation.FuncAnimation(fig, rotar, frames=range(0, 360, 2),
                               interval=40, blit=False)

# Para guardar como gif descomenta:
# ani.save('orbitas_3d.gif', writer='pillow', fps=25, dpi=100)

plt.tight_layout()
plt.show()